In [0]:
# ============================================================
# 1_topic_tagging_v5_classify
# LLM 배치 분류 (mini 모델 + 동적 샘플링 + 전반적 rule pre-filter)
# ============================================================
# 이 노트북은 1_topic_tagging_v5 에서 생성된 rule profile / topic pool을 기반으로
# 미완료 그룹의 메모를 분류하는 전용 classify 노트북입니다.
#
# 핵심 변경점:
#  1. 모델: databricks-gpt-5-4-mini (비용 절감)
#  2. 동적 샘플링: <1K 전수, 1K~10K 10%, ≥10K 1000건
#  3. "전반적" 그룹: 감정 전용 메모 rule pre-filter 후 샘플링
#  4. 프롬프트에 representative_memos 예시 포함
# ============================================================

from __future__ import annotations

import json
import re
import time
from typing import Any, Dict, List, Optional

import pandas as pd
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

# --------------------------------------------------
# LLM Endpoint
# --------------------------------------------------
ENDPOINT = "databricks-gpt-5-4-mini"

# --------------------------------------------------
# DB / Tables
# --------------------------------------------------
PROMPT_VERSION = "category_topic_dynamic_rules_allgroups_v1"
SAVE_DB = "sandbox.z_jungryo_lee"

SOURCE_TABLE = "kic_data_ods.buzzmetrix.buzzmetrix"

RULE_PROFILE_TABLE  = f"{SAVE_DB}.category_topic_rule_profile_{PROMPT_VERSION}"
TOPIC_POOL_TABLE    = f"{SAVE_DB}.category_topic_pool_{PROMPT_VERSION}"
DETAIL_TABLE        = f"{SAVE_DB}.category_topic_detail_{PROMPT_VERSION}"
SUMMARY_TABLE       = f"{SAVE_DB}.category_topic_summary_{PROMPT_VERSION}"
PROGRESS_TABLE      = f"{SAVE_DB}.category_topic_progress_{PROMPT_VERSION}"
FAILED_GROUP_TABLE  = f"{SAVE_DB}.category_topic_failed_group_{PROMPT_VERSION}"
GROUP_STATUS_TABLE  = f"{SAVE_DB}.category_topic_group_status"

# --------------------------------------------------
# Thresholds / Limits
# --------------------------------------------------
SAMPLE_TIER_FULL    = 1000     # 미만이면 전수
SAMPLE_TIER_PERCENT = 10000    # 미만이면 10%
SAMPLE_PERCENT      = 0.10     # 10%
SAMPLE_CAP          = 1000     # 상한

CLASSIFY_BATCH_SIZE  = 25
MIN_ROWS_PER_TOPIC   = 10
MAX_TOKENS           = 2200
MAX_RETRIES          = 3
BACKOFF_BASE         = 1.8
RATE_LIMIT_SECONDS   = 0.4
RESUME_FROM_CHECKPOINT = True

print("[CONFIG] Loaded")
print(f"  ENDPOINT            = {ENDPOINT}")
print(f"  SOURCE_TABLE        = {SOURCE_TABLE}")
print(f"  DETAIL_TABLE        = {DETAIL_TABLE}")
print(f"  Sampling: <{SAMPLE_TIER_FULL} 전수 | <{SAMPLE_TIER_PERCENT} {int(SAMPLE_PERCENT*100)}% | ≥{SAMPLE_TIER_PERCENT} {SAMPLE_CAP}건")
print(f"  CLASSIFY_BATCH_SIZE = {CLASSIFY_BATCH_SIZE}")

In [0]:
# ============================================================
# 2) Helpers & Checkpoint
# ============================================================

def clean_text(x: Any) -> str:
    return "" if x is None else re.sub(r"\s+", " ", str(x)).strip()


def extract_json(text: str) -> Dict[str, Any]:
    text = clean_text(text)
    try:
        return json.loads(text)
    except Exception:
        pass
    match = re.search(r"\{.*\}", text, flags=re.S)
    if match:
        candidate = match.group(0)
        try:
            return json.loads(candidate)
        except Exception:
            candidate = re.sub(r",\s*}", "}", candidate)
            candidate = re.sub(r",\s*]", "]", candidate)
            return json.loads(candidate)
    raise ValueError(f"Invalid JSON: {text[:1000]}")


def chunk_list(items: List[Any], size: int) -> List[List[Any]]:
    return [items[i:i + size] for i in range(0, len(items), size)]


def sc_label(sc_measurement: int) -> str:
    return {1: "긍정", -1: "부정"}.get(sc_measurement, "기타")


def overall_label(sc_measurement: int) -> str:
    return {1: "전반적 긍정", -1: "전반적 부정"}.get(sc_measurement, "전반적 평가")


def call_llm(messages: List[Dict[str, str]], max_tokens: int = MAX_TOKENS) -> Dict[str, Any]:
    from mlflow.deployments import get_deploy_client
    client = get_deploy_client("databricks")
    payload = {"messages": messages, "temperature": 0.0, "max_tokens": max_tokens}
    last_error: Optional[Exception] = None
    for attempt in range(MAX_RETRIES):
        resp = None
        try:
            resp = client.predict(endpoint=ENDPOINT, inputs=payload)
            if isinstance(resp, dict):
                if "choices" in resp and resp["choices"]:
                    return extract_json(resp["choices"][0]["message"]["content"])
                if "predictions" in resp and resp["predictions"]:
                    pred0 = resp["predictions"][0]
                    if isinstance(pred0, dict) and "content" in pred0:
                        return extract_json(pred0["content"])
                    if isinstance(pred0, str):
                        return extract_json(pred0)
                if "content" in resp:
                    return extract_json(resp["content"])
            if isinstance(resp, str):
                return extract_json(resp)
            raise ValueError(f"Unexpected response schema: {resp}")
        except Exception as exc:
            last_error = exc
            print(f"[LLM ERROR] attempt={attempt + 1}/{MAX_RETRIES}, error={repr(exc)}")
            time.sleep(BACKOFF_BASE ** attempt)
    raise RuntimeError(f"LLM call failed: {repr(last_error)}")


# --------------------------------------------------
# Checkpoint / Resume Helpers
# --------------------------------------------------
PROGRESS_STRUCT = T.StructType([
    T.StructField("cate_1_depth", T.StringType(), True),
    T.StructField("cate_2_depth", T.StringType(), True),
    T.StructField("sc_measurement", T.IntegerType(), True),
    T.StructField("stage", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("message", T.StringType(), True),
    T.StructField("event_ts", T.TimestampType(), True),
])

FAILED_STRUCT = T.StructType([
    T.StructField("cate_1_depth", T.StringType(), True),
    T.StructField("cate_2_depth", T.StringType(), True),
    T.StructField("sc_measurement", T.IntegerType(), True),
    T.StructField("stage", T.StringType(), True),
    T.StructField("error_message", T.StringType(), True),
    T.StructField("event_ts", T.TimestampType(), True),
])


def append_table(df, table_name: str) -> None:
    if spark.catalog.tableExists(table_name):
        df.write.mode("append").format("delta").saveAsTable(table_name)
    else:
        df.write.mode("overwrite").format("delta").saveAsTable(table_name)


def log_progress(cate_1_depth, cate_2_depth, sc_measurement, stage, status, message=""):
    row = [(
        clean_text(cate_1_depth), clean_text(cate_2_depth), int(sc_measurement),
        clean_text(stage), clean_text(status), clean_text(message),
        pd.Timestamp.now().to_pydatetime(),
    )]
    append_table(spark.createDataFrame(row, schema=PROGRESS_STRUCT), PROGRESS_TABLE)


def log_failure(cate_1_depth, cate_2_depth, sc_measurement, stage, error_message):
    row = [(
        clean_text(cate_1_depth), clean_text(cate_2_depth), int(sc_measurement),
        clean_text(stage), clean_text(error_message),
        pd.Timestamp.now().to_pydatetime(),
    )]
    append_table(spark.createDataFrame(row, schema=FAILED_STRUCT), FAILED_GROUP_TABLE)


def get_done_group_keys(stage: str):
    if (not RESUME_FROM_CHECKPOINT) or (not spark.catalog.tableExists(PROGRESS_TABLE)):
        return set()
    rows = (
        spark.table(PROGRESS_TABLE)
        .where(F.col("stage") == F.lit(stage))
        .where(F.col("status") == F.lit("done"))
        .select("cate_1_depth", "cate_2_depth", "sc_measurement")
        .distinct()
        .collect()
    )
    return {(r["cate_1_depth"], r["cate_2_depth"], int(r["sc_measurement"])) for r in rows}


def delete_group_rows(table_name, cate_1_depth, cate_2_depth, sc_measurement):
    if not spark.catalog.tableExists(table_name):
        return
    spark.sql(f"""
        delete from {table_name}
        where cate_1_depth = '{cate_1_depth}'
          and cate_2_depth = '{cate_2_depth}'
          and sc_measurement = {sc_measurement}
    """)


# --------------------------------------------------
# 동적 샘플링 helpers
# --------------------------------------------------
def get_group_total_count(cate_1_depth: str, cate_2_depth: str, sc_measurement: int) -> int:
    query = f"""
    select count(distinct memo) as cnt
    from {SOURCE_TABLE}
    where cate_1_depth = '{cate_1_depth}'
      and cate_2_depth = '{cate_2_depth}'
      and sc_measurement = {sc_measurement}
      and memo is not null
      and length(trim(memo)) > 0
    """
    return spark.sql(query).collect()[0]["cnt"]


def get_dynamic_sample_size(total_count: int) -> int:
    if total_count < SAMPLE_TIER_FULL:
        return total_count
    elif total_count < SAMPLE_TIER_PERCENT:
        return max(int(total_count * SAMPLE_PERCENT), 1)
    else:
        return SAMPLE_CAP


def load_group_sample_df(cate_1_depth, cate_2_depth, sc_measurement, max_rows):
    query = f"""
    with base as (
        select distinct memo
        from {SOURCE_TABLE}
        where 1=1
          and cate_1_depth = '{cate_1_depth}'
          and cate_2_depth = '{cate_2_depth}'
          and sc_measurement = {sc_measurement}
          and memo is not null
          and length(trim(memo)) > 0
    ),
    sampled as (
        select
            memo,
            row_number() over (
                order by md5(
                    concat(
                        coalesce(memo, ''),
                        '||', '{cate_1_depth}',
                        '||', '{cate_2_depth}',
                        '||', cast({sc_measurement} as string),
                        '||seed_20260420'
                    )
                )
            ) as rn
        from base
    )
    select memo
    from sampled
    where rn <= {max_rows}
    order by rn
    """
    return spark.sql(query).withColumn("_row_id", F.monotonically_increasing_id()).cache()


def load_group_all_memos_df(cate_1_depth: str, cate_2_depth: str, sc_measurement: int):
    query = f"""
    select distinct memo
    from {SOURCE_TABLE}
    where cate_1_depth = '{cate_1_depth}'
      and cate_2_depth = '{cate_2_depth}'
      and sc_measurement = {sc_measurement}
      and memo is not null
      and length(trim(memo)) > 0
    """
    return spark.sql(query)


print("[OK] Helpers & Checkpoint loaded")

In [0]:
# ============================================================
# 3) Load Pending (Target) Groups
# ============================================================

progress_df = spark.table(GROUP_STATUS_TABLE)

target_groups = (
    progress_df
    .where("is_done = 0")
    .select("cate_1_depth", "cate_2_depth", "sc_measurement")
    .distinct()
    .collect()
)

print(f"[INFO] Total pending groups = {len(target_groups)}")

display(
    progress_df
    .where("is_done = 0")
    .orderBy("cate_1_depth", "cate_2_depth", "sc_measurement")
)

In [0]:
# ============================================================
# 4) Category Feature Patterns (v5 동일)
# ============================================================

COMMON_FEATURE_PATTERNS = [
    "quality", "picture", "image", "visual", "sound", "audio", "video", "screen", "display", "tv",
    "setup", "install", "installation", "manual", "guide", "weight", "stand", "mount", "wall",
    "cable", "port", "hdmi", "bluetooth", "wifi", "wireless", "connect", "connection", "compatibility",
    "os", "software", "menu", "ui", "app", "apps", "channel", "content", "voice", "game", "gaming",
    "iot", "mobile", "brand", "service", "support", "warranty", "energy", "efficiency", "price",
    "value", "design", "color", "material", "finish", "heat", "durability", "panel", "glare",
    "angle", "brightness", "contrast", "resolution", "sharp", "clarity", "bass", "surround",
    "화질", "음질", "사운드", "화면", "디스플레이", "설치", "세팅", "매뉴얼", "가이드", "무게",
    "스탠드", "벽걸이", "선정리", "단자", "블루투스", "와이파이", "연결", "호환", "소프트웨어",
    "메뉴", "ui", "앱", "채널", "콘텐츠", "음성", "게임", "iot", "모바일", "브랜드", "서비스",
    "보증", "에너지", "효율", "가격", "가성비", "디자인", "색상", "소재", "마감", "발열",
    "내구성", "패널", "반사", "시야각", "밝기", "명암", "해상도", "선명", "저음", "서라운드"
]

CATEGORY_FEATURE_PATTERNS = {
    ("01. 사이즈", "01-01. TV 사이즈"): [
        "size", "inch", "inches", "big", "large", "small", "fit", "fits",
        "사이즈", "크기", "인치", "대형", "소형", "공간", "잘맞"
    ],
    ("02. 화질", "02-01. 선명도"): [
        "sharp", "sharpness", "clear", "clarity", "crisp", "blur", "blurry",
        "선명", "선명도", "또렷", "흐림", "블러"
    ],
    ("02. 화질", "02-02. 컴러"): [
        "color", "colour", "vivid", "vibrant", "lifelike", "saturation",
        "컴러", "색감", "생생", "채도"
    ],
    ("02. 화질", "02-03. 밝기"): [
        "bright", "brightness", "dim", "luminous",
        "밝기", "밝음", "어두움"
    ],
    ("02. 화질", "02-04. 명암비"): [
        "contrast", "black level", "black", "white balance",
        "명암", "명암비", "블랙", "검정"
    ],
    ("02. 화질", "02-05. 해상도"): [
        "resolution", "4k", "uhd", "hdr", "detail", "pixel",
        "해상도", "4k", "uhd", "hdr", "디테일", "픽셀"
    ],
    ("02. 화질", "02-06. 움직이는 영상 표현"): [
        "motion", "smooth", "blur", "judder", "stutter", "fast scene",
        "움직임", "모션", "부드러움", "잔상", "버벡"
    ],
    ("02. 화질", "02-08. 시야각"): [
        "angle", "viewing angle", "off angle",
        "시야각", "각도"
    ],
    ("02. 화질", "02-09. 반사율"): [
        "glare", "reflection", "reflective", "anti glare",
        "반사", "반사율", "빛반사", "눈부심"
    ],
    ("02. 화질", "02-10. 화질세팅"): [
        "setting", "mode", "calibration", "preset",
        "설정", "세팅", "모드", "보정"
    ],
    ("02. 화질", "02-10. 화질세팅_(1)화질 모드"): [
        "picture mode", "mode", "preset",
        "화질모드", "모드", "프리셋"
    ],
    ("02. 화질", "02-20. 전반적 화질"): [
        "picture", "image", "visual", "screen quality",
        "화질", "화면", "영상"
    ],
    ("03. 음질", "03-01. 출력"): [
        "volume", "loud", "output", "speaker power",
        "출력", "볼륨", "소리크기"
    ],
    ("03. 음질", "03-02. 선명한 음질"): [
        "clear sound", "clarity", "crisp audio",
        "선명", "맑음", "또렷"
    ],
    ("03. 음질", "03-03. 음질 세팅"): [
        "sound mode", "setting", "equalizer", "eq",
        "설정", "세팅", "모드", "eq"
    ],
    ("03. 음질", "03-04. 서라운드"): [
        "surround", "immersive", "spatial",
        "서라운드", "공간감"
    ],
    ("03. 음질", "03-05. 저음/베이스"): [
        "bass", "low end", "deep sound",
        "저음", "베이스"
    ],
    ("03. 음질", "03-20. 전반적 음질"): [
        "sound", "audio", "speaker",
        "소리", "음질", "사운드"
    ],
    ("04. 디자인", "04-01. 옆면 두께"): [
        "thin", "thickness", "slim",
        "얇음", "두께", "슬림"
    ],
    ("04. 디자인", "04-02. 베젤(프레임) 두께"): [
        "bezel", "frame",
        "베젤", "프레임"
    ],
    ("04. 디자인", "04-03. 스탠드 높이/형태"): [
        "stand", "height", "base",
        "스탠드", "높이", "받침"
    ],
    ("04. 디자인", "04-04. 벽걸이 디자인"): [
        "wall mount", "mount", "flush",
        "벽걸이", "마운트"
    ],
    ("04. 디자인", "04-05. 소재"): [
        "material", "texture",
        "소재", "재질"
    ],
    ("04. 디자인", "04-06. 색상"): [
        "color", "colour",
        "색상", "색"
    ],
    ("04. 디자인", "04-07. 마감"): [
        "finish", "build quality",
        "마감", "완성도"
    ],
    ("04. 디자인", "04-08. 후면부 디자인"): [
        "rear", "back design", "back panel",
        "후면", "뒷면"
    ],
    ("04. 디자인", "04-20. 전반적 디자인"): [
        "beautiful", "stylish", "look", "appearance",
        "예쁨", "외관", "디자인"
    ],
    ("05. 설치/세팅", "05-01. 세팅"): [
        "setup", "set up", "configure",
        "설치", "세팅", "설정"
    ],
    ("05. 설치/세팅", "05-02. 매뉴얼/가이드"): [
        "manual", "guide", "instruction",
        "매뉴얼", "가이드", "설명서"
    ],
    ("05. 설치/세팅", "05-03. 무게"): [
        "weight", "heavy", "light",
        "무게", "무겁", "가벼"
    ],
    ("05. 설치/세팅", "05-04. 선처리"): [
        "cable", "wire", "cable management",
        "선정리", "케이블", "선"
    ],
    ("05. 설치/세팅", "05-05. 각도 조절(벽걸이)"): [
        "angle", "tilt", "swivel",
        "각도", "틸트", "회전"
    ],
    ("05. 설치/세팅", "05-06. 벽걸이 설치용이성"): [
        "wall mount", "mounting",
        "벽걸이", "설치"
    ],
    ("05. 설치/세팅", "05-07. 스탠드 설치용이성"): [
        "stand assembly", "stand install",
        "스탠드", "조립"
    ],
    ("05. 설치/세팅", "05-20. 전반적 설치용이성"): [
        "install", "installation", "easy setup",
        "설치", "세팅", "조립"
    ],
    ("06. 연결성", "06-01. 연결기기 호환성"): [
        "compatible", "compatibility", "device", "devices",
        "호환", "호환성", "기기"
    ],
    ("06. 연결성", "06-02. 무선 연결성"): [
        "wifi", "wireless", "bluetooth", "pairing",
        "와이파이", "무선", "블루투스", "페어링"
    ],
    ("06. 연결성", "06-03. 연결단자 지원/개수"): [
        "hdmi", "usb", "port", "ports",
        "단자", "포트", "hdmi", "usb"
    ],
    ("06. 연결성", "06-04. (편리한)단자 위치"): [
        "port location", "easy access",
        "단자위치", "접근"
    ],
    ("06. 연결성", "06-20. 전반적 연결성"): [
        "connect", "connection",
        "연결", "연결성"
    ],
    ("07. 스마트 사용성", "07-01. 채널/컨텐츠 APP"): [
        "app", "apps", "channel", "content", "streaming", "ott",
        "앱", "채널", "콘텐츠", "스트리밍", "ott"
    ],
    ("07. 스마트 사용성", "07-02. 구동성/구동속도"): [
        "fast", "slow", "lag", "speed", "loading",
        "속도", "빠름", "느림", "로딩", "렉"
    ],
    ("07. 스마트 사용성", "07-02. 구동성/구동속도_(1)TV 전반"): [
        "fast", "slow", "lag", "speed", "tv response",
        "속도", "빠름", "느림", "반응", "렉"
    ],
    ("07. 스마트 사용성", "07-02. 구동성/구동속도_(2)APP/웹전반"): [
        "app speed", "web speed", "lag", "loading",
        "앱속도", "웹속도", "로딩", "렉"
    ],
    ("07. 스마트 사용성", "07-03. 메뉴 구성/UI"): [
        "menu", "ui", "interface", "navigation",
        "메뉴", "ui", "인터페이스", "탐색"
    ],
    ("07. 스마트 사용성", "07-04. SW/OS"): [
        "os", "software", "update", "bug",
        "os", "소프트웨어", "업데이트", "버그"
    ],
    ("07. 스마트 사용성", "07-05. 컨텐츠 탐색 사용성"): [
        "search", "browse", "discover",
        "탐색", "검색", "브라우징"
    ],
    ("07. 스마트 사용성", "07-06. 리모컨 사용성"): [
        "remote", "control", "button", "layout", "pointer", "backlight", "easy",
        "리모컨", "조작", "버튼", "레이아웃", "포인터", "백라이트", "편리"
    ],
    ("07. 스마트 사용성", "07-07. 리모컨 디자인"): [
        "remote design", "remote look",
        "리모컨 디자인", "외관"
    ],
    ("07. 스마트 사용성", "07-08. 음성 인식/조작"): [
        "voice", "speech", "assistant",
        "음성", "음성인식", "음성조작"
    ],
    ("07. 스마트 사용성", "07-09. 게임 기능"): [
        "game", "gaming", "latency",
        "게임", "게이밍", "지연"
    ],
    ("07. 스마트 사용성", "07-10. 부가 기능"): [
        "feature", "features", "extra function",
        "기능", "부가기능"
    ],
    ("07. 스마트 사용성", "07-11. 홈 IoT"): [
        "iot", "smart home",
        "iot", "스마트홈"
    ],
    ("07. 스마트 사용성", "07-12. 모바일 연동"): [
        "mobile", "phone", "cast", "mirror",
        "모바일", "휴대폰", "캐스트", "미러링"
    ],
    ("07. 스마트 사용성", "07-13. 광고"): [
        "ad", "ads", "advertisement",
        "광고"
    ],
    ("07. 스마트 사용성", "07-20. 전반적 스마트 사용성"): [
        "smart", "usability", "easy to use",
        "스마트", "사용성", "편리"
    ],
    ("08. 내구성", "08-01. A/S"): [
        "service", "support", "repair", "warranty",
        "as", "서비스", "지원", "수리", "보증"
    ],
    ("08. 내구성", "08-02. 품질보증기간"): [
        "warranty", "guarantee",
        "보증", "보증기간"
    ],
    ("08. 내구성", "08-03. 잔상"): [
        "ghosting", "burn in", "image retention",
        "잔상", "번인"
    ],
    ("08. 내구성", "08-04. 패널 내구성"): [
        "panel", "durability", "screen life",
        "패널", "내구성"
    ],
    ("08. 내구성", "08-05. 발열"): [
        "heat", "heating", "hot",
        "발열", "뜨거움"
    ],
    ("08. 내구성", "08-20. 전반적 내구성"): [
        "durable", "reliable", "lasting",
        "내구성", "튼튼", "오래감"
    ],
    ("09. 친환경", "09-01. 에너지 효율"): [
        "energy", "efficient", "efficiency", "power saving",
        "에너지", "효율", "절전"
    ],
    ("09. 친환경", "09-02. 친환경 소재"): [
        "eco", "eco-friendly", "material",
        "친환경", "소재"
    ],
    ("10. 가격", "10-01. 가격/가격대비 가치"): [
        "price", "value", "worth", "deal", "budget", "affordable",
        "가격", "가성비", "값어치", "딜", "저렴"
    ],
    ("11. 브랜드", "11-01. 브랜드"): [
        "brand", "samsung", "lg", "sony", "tcl", "hisense",
        "브랜드", "삼성", "엘지", "소니"
    ],
    ("12. 품질 불량", "12-01. 화질 불량"): [
        "defect", "screen issue", "dead pixel", "dark", "blurry",
        "불량", "화질불량", "데드픽셀", "암부", "흐림"
    ],
    ("12. 품질 불량", "12-02. 제품 불량"): [
        "defective", "broken", "fault", "problem",
        "불량", "고장", "문제"
    ],
    ("12. 품질 불량", "12-03. 설치 불량"): [
        "installation issue", "bad install",
        "설치불량", "설치문제"
    ],
    ("12. 품질 불량", "12-04. 기능 불량"): [
        "malfunction", "doesn't work", "failed", "not working",
        "기능불량", "작동안함", "먹통", "고장"
    ],
    ("12. 품질 불량", "12-05. 기타 불량"): [
        "issue", "problem", "fault",
        "불량", "문제", "이슈"
    ],
    ("13. 전반적 평가", "13-01. 전반적 평가"): [
        "overall", "generally", "satisfied", "happy", "purchase", "product",
        "전반적", "전체적", "만족", "구매", "제품"
    ],
}


def get_feature_patterns(cate_1_depth: str, cate_2_depth: str) -> List[str]:
    return COMMON_FEATURE_PATTERNS + CATEGORY_FEATURE_PATTERNS.get((cate_1_depth, cate_2_depth), [])


def has_specific_reason_for_category(text: str, clue_keywords: List[str], cate_1_depth: str, cate_2_depth: str) -> bool:
    memo = clean_text(text).lower()
    if any(clean_text(k).lower() in memo for k in clue_keywords if clean_text(k)):
        return True
    feature_patterns = get_feature_patterns(cate_1_depth, cate_2_depth)
    return any(clean_text(p).lower() in memo for p in feature_patterns if clean_text(p))


def should_be_overall_for_category(text: str, clue_keywords: List[str], generic_terms: List[str], cate_1_depth: str, cate_2_depth: str) -> bool:
    memo = clean_text(text).lower()
    if not memo:
        return False
    if len(memo) > 14:
        return False
    if has_specific_reason_for_category(memo, clue_keywords, cate_1_depth, cate_2_depth):
        return False
    if len(re.findall(r"[A-Za-z\uac00-\ud7a3]+", memo)) > 3:
        return False
    return any(clean_text(t).lower() in memo for t in generic_terms if clean_text(t))


# ============================================================
# Overall Sentiment Detection  ("전반적" 그룹 전용 rule pre-filter)
# ============================================================
# 전반적 긍정/부정 = length<25 + 감정 표현만 + 피처 설명 단어 없음

# 감정 전용 표현 (형용사 / 부사)
SENTIMENT_EXPRESSIONS = {
    # --- Korean ---
    "좋아요", "좋아", "좋음", "좋다", "좋습니다", "좋네요", "좋았", "좋은",
    "최고", "만족", "대만족", "훌륭", "완벽", "추천", "강추", "굿", "굿굿", 
    "괜찮", "맘에", "마음에", "만족스러"
    "별로", "나쁨", "나빠", "나쁜", "불만", "실망", "후회", "최악",
    "싫어", "아쉬", "그저그래", "그냥", "보통",
    "너무", "정말", "아주", "매우", "진짜",
    # --- English ---
    "good", "great", "excellent", "awesome", "amazing", "perfect",
    "wonderful", "fantastic", "love", "loved", "best", "nice",
    "superb", "outstanding", "incredible", "brilliant",
    "bad", "terrible", "awful", "horrible", "worst", "poor",
    "suck", "sucks", "disappointed", "disappointing", "hate", "ugly",
    "useless", "okay", "ok", "fine", "not bad", "rubbish", "crap",
    "very", "really", "so", "super", "quite", "absolutely", "totally",
    "highly", "extremely", "pretty", "just",
    # --- German ---
    "gut", "toll", "super", "perfekt", "schlecht", "ausgezeichnet", "sehr",
    # --- Spanish ---
    "bueno", "malo", "excelente", "genial", "perfecto", "muy",
}

# Stopwords / filler (의미 없는 단어 - 무시 가능)
FILLER_WORDS = {
    "it", "is", "was", "the", "a", "an", "i", "my", "this", "that",
    "its", "im", "am", "are", "be", "been", "being",
    "not", "no", "yes", "yeah", "yep", "nope",
    "에요", "이에요", "입니다", "해요", "요", "네요",
    "다", "은", "는", "이", "가", "를", "을", "도", "만",
    "데", "에", "서", "로",
}

# 피처/설명적 단어 (이 단어가 있으면 전반적 긍정/부정이 아님)
FEATURE_EXCLUSION_WORDS = [
    # Size
    "크", "작", "커", "작아", "big", "small", "large", "tiny", "huge",
    # Brightness / Darkness
    "밝", "어두", "밝아", "어두워", "bright", "dark", "dim",
    # Sound volume
    "시끄러", "조용", "loud", "quiet", "silent",
    # Speed
    "빠르", "느리", "fast", "slow",
    # Weight
    "무겁", "가벼", "heavy", "light",
    # Sharpness
    "선명", "흐릿", "sharp", "clear", "blurry",
    # Thickness
    "얇", "두꺼", "thin", "thick",
    # Color
    "색감", "색상", "color", "colour",
    # Feature categories
    "화질", "음질", "화면", "소리", "사운드", "디자인", "설치",
    "리모컨", "앱", "게임", "연결", "wifi", "블루투스", "hdmi",
    "패널", "베젤", "스탠드", "가격", "배송", "보증",
    "picture", "sound", "audio", "screen", "display", "remote",
    "price", "delivery", "install", "panel", "bezel",
]


def is_pure_overall_sentiment(memo: str) -> bool:
    """
    Rule-based: 전반적 긍정/부정 판단.
    - len < 25
    - 감정 표현만으로 구성 (종결어알 · 조동사 제외, 감정 형용사/부사만)
    - 피처 설명 단어(크다/작다/밝다/어둡다 등) 포함 시 False
    """
    if not memo:
        return False
    text = memo.strip()
    if len(text) >= 25:
        return False

    text_lower = text.lower()

    # 피처/설명 단어가 하나라도 있으면 → NOT overall
    for w in FEATURE_EXCLUSION_WORDS:
        if w.lower() in text_lower:
            return False

    # 토큰화 후 감정+filler 만으로 구성되는지 확인
    tokens = re.findall(r"[a-z\uac00-\ud7a3]+", text_lower)
    if not tokens:
        return False

    has_sentiment = False
    for tok in tokens:
        if tok in SENTIMENT_EXPRESSIONS:
            has_sentiment = True
        elif tok in FILLER_WORDS:
            continue
        else:
            # 감정도 filler도 아닌 단어 → 구체적 내용 있음
            return False

    return has_sentiment


print("[OK] Category feature patterns + overall sentiment rule loaded")

In [0]:
# ============================================================
# 4.1) SENTIMENT_EXPRESSIONS / FILLER_WORDS 확장
# ============================================================
# 테스트에서 발견된 미등록 변형 + 일반 VOC 상용어 추가

SENTIMENT_EXPRESSIONS.update({
    # --- Korean 변형/슠랭 ---
    "굿굿", "구욷", "구웃", "구드",               # 굿 변형
    "짱", "쥐", "대박",                          # 감탄
    "좋지", "좋죠", "좋네", "좋구나", "좋군", "좋더", "좋는데",  # 좋- 활용형
    "괜찮은", "괜찮네", "괜찮다",
    "훌륭해", "완벽해", "훌륭함", "완벽함",
    "만족해", "만족함",
    "맘에들", "맘에듬",
    "감사", "고맙",
    "사랑", "사랑해",
    "별로야", "구리", "구림",                    # 부정 변형
    "보통이", "그랭그랭",
    # --- Korean 강조 접두사 ---
    "겹나", "존나", "완전",
    # --- English 추가 ---
    "satisfied", "happy", "pleased", "impressed",
    "recommended", "recommend",
    "worth", "decent", "solid", "wonderful",
    "like", "liked", "liking", "likes", "loves",
    "dislike", "garbage", "waste", "regret",
    "better", "worse",
    "premium", "top",
    # --- German 추가 ---
    "wunderbar", "fantastisch", "zufrieden",
    # --- Spanish 추가 ---
    "satisfecho", "terrible", "horrible",
})

FILLER_WORDS.update({
    # --- 제품명/일반 명사 (VOC 맥락에서 의미 없는 단어) ---
    "tv", "product", "purchase", "buy", "bought",
    "upgrade", "item", "unit", "set", "model",
    "thing", "one",
    # --- 대명사 / 인칭 ---
    "he", "she", "we", "they", "me", "you",
    "her", "him", "them", "our", "your", "their",
    # --- 접속사 / 전치사 ---
    "of", "for", "with", "and", "but", "or",
    "to", "in", "on", "at", "by", "from", "so", "too",
    "also", "as", "than", "far",
    # --- 기타 ---
    "all", "what", "how", "pro", "pros", "con", "cons",
    "smart", "new", "old",
    # --- Korean 어미/조사 ---
    "이거", "그거", "정말로", "전체적으로",
    "어요", "아요", "이란",
})

print(f"[OK] SENTIMENT_EXPRESSIONS expanded to {len(SENTIMENT_EXPRESSIONS)} entries")
print(f"[OK] FILLER_WORDS expanded to {len(FILLER_WORDS)} entries")

In [0]:
# ============================================================
# 4.5) Group Reference Words & Unified Trivial Memo Filter
# ============================================================
# 모든 그룹에 적용:
#   그룹 지칭 단어 + 감정 표현만으로 구성된 메모는
#   rule 기반으로 사전 필터링 후 샘플링합니다.
#   예: "리모컨 좋아요", "remote good", "sound great"
# ============================================================

def get_group_reference_words(
    cate_1_depth: str,
    cate_2_depth: str,
    clue_keywords: List[str] = None,
) -> set:
    """
    그룹 지칭 단어 추출:
    1. CATEGORY_FEATURE_PATTERNS 에서
    2. cate_2_depth 이름에서 (예: "07-06. 리모컨 사용성" → "리모컨", "사용성")
    3. clue_keywords (rule_profile) 에서
    """
    words: set = set()

    # 1. CATEGORY_FEATURE_PATTERNS
    cat_patterns = CATEGORY_FEATURE_PATTERNS.get(
        (cate_1_depth, cate_2_depth), []
    )
    words.update(w.lower() for w in cat_patterns if len(w) >= 2)

    # 2. cate_2_depth 이름에서 추출 (번호/괄호 제거)
    name_part = re.sub(r"^\d+[-_.\s]*", "", cate_2_depth).strip()
    name_part = re.sub(r"\(.*?\)", "", name_part).strip()
    name_part = re.sub(r"_\(\d+\).*$", "", name_part).strip()
    for token in re.findall(r"[a-zA-Z\uac00-\ud7a3]{2,}", name_part):
        words.add(token.lower())

    # 3. clue_keywords (rule_profile)
    if clue_keywords:
        for kw in clue_keywords:
            kw_clean = clean_text(kw).lower()
            if len(kw_clean) >= 2:
                words.add(kw_clean)

    return words


def is_group_sentiment_only(memo: str, group_ref_words: set) -> bool:
    """
    그룹 지칭 단어 + 감정 표현만으로 구성된 메모 판단.
    조건: len < 25, 토큰이 모두 (감정 | filler | 그룹지칭)

    주의: FEATURE_EXCLUSION_WORDS 체크 시,
    그룹 지칭 단어와 겹치는 단어는 제외합니다.
    예: 리모컨 그룹에서 "remote" → 지칭 단어로 취급, exclusion 아님
    """
    if not memo:
        return False
    text = memo.strip()
    if len(text) >= 25:
        return False

    text_lower = text.lower()

    # 피처 설명 단어 체크 (그룹 지칭 단어와 겹치는 것은 제외)
    for w in FEATURE_EXCLUSION_WORDS:
        wl = w.lower()
        if wl in text_lower and wl not in group_ref_words:
            return False

    tokens = re.findall(r"[a-z\uac00-\ud7a3]+", text_lower)
    if not tokens:
        return False

    has_sentiment = False
    has_group_ref = False

    for tok in tokens:
        if tok in SENTIMENT_EXPRESSIONS:
            has_sentiment = True
        elif tok in FILLER_WORDS:
            continue
        elif tok in group_ref_words:
            has_group_ref = True
        else:
            # 감정도 filler도 그룹지칭도 아닌 단어 → 구체적 내용
            return False

    return has_sentiment and has_group_ref


def is_trivial_memo(memo: str, group_ref_words: set = None) -> bool:
    """
    사전 필터링 대상 메모 판단 (모든 그룹 공통):
    1. 순수 감정 표현 — "좋아요", "great"
    2. 그룹 지칭 + 감정 표현 — "리모컨 좋아요", "remote good"
    """
    if is_pure_overall_sentiment(memo):
        return True
    if group_ref_words and is_group_sentiment_only(memo, group_ref_words):
        return True
    return False


print("[OK] Group reference words & trivial memo filter loaded")

In [0]:
# ============================================================
# [TEST] Pre-filter 결과 확인 - 리모컨 사용성 / 전반적 평가 / 가격
# ============================================================

# rule_profile에서 clue_keywords 로드
rule_profile_df = spark.table(RULE_PROFILE_TABLE)

test_groups = [
    ("07. 스마트 사용성", "07-06. 리모컨 사용성", 1),
    ("13. 전반적 평가", "13-01. 전반적 평가", 1),
    ("10. 가격", "10-01. 가격/가격대비 가치", 1),
]

for c1, c2, sc in test_groups:
    print(f"\n{'='*60}")
    print(f"GROUP: {c1} / {c2} / sc={sc}")
    print(f"{'='*60}")

    # clue_keywords 추출
    rp_row = (
        rule_profile_df
        .where(
            (F.col("cate_1_depth") == c1)
            & (F.col("cate_2_depth") == c2)
            & (F.col("sc_measurement") == sc)
        )
        .first()
    )
    clue_kw = []
    if rp_row and rp_row["clue_keywords_json"]:
        try:
            clue_kw = json.loads(rp_row["clue_keywords_json"])
        except Exception:
            pass

    # 그룹 지칭 단어 추출
    ref_words = get_group_reference_words(c1, c2, clue_kw)
    print(f"[REF WORDS] ({len(ref_words)}개): {sorted(list(ref_words))[:20]}")

    # 전체 메모 로드
    all_df = load_group_all_memos_df(c1, c2, sc)
    total = all_df.count()

    # UDF 적용
    _ref = ref_words
    _udf = F.udf(lambda m: is_trivial_memo(m, _ref), T.BooleanType())
    tagged = all_df.withColumn("_trivial", _udf(F.col("memo")))

    trivial_cnt = tagged.where("_trivial = true").count()
    non_trivial_cnt = tagged.where("_trivial = false").count()

    print(f"[COUNTS] total={total}, trivial={trivial_cnt} ({round(trivial_cnt/total*100,1)}%), non_trivial={non_trivial_cnt}")

    # trivial 예시 메모
    print(f"\n[필터된 trivial 메모 상위 10건]")
    trivial_samples = (
        tagged.where("_trivial = true")
        .select("memo")
        .limit(10)
        .collect()
    )
    for i, r in enumerate(trivial_samples, 1):
        print(f"  {i}. {r['memo']}")

    # non-trivial 예시 메모 (짧은 것 위주)
    print(f"\n[남은 non-trivial 메모 상위 10건 (짧은순)]")
    non_trivial_samples = (
        tagged.where("_trivial = false")
        .withColumn("_len", F.length(F.col("memo")))
        .orderBy("_len")
        .select("memo")
        .limit(10)
        .collect()
    )
    for i, r in enumerate(non_trivial_samples, 1):
        print(f"  {i}. {r['memo']}")

In [0]:
# ============================================================
# 5) Classify All Groups (Unified Pre-filter)
#    - 모든 그룹: trivial memo rule pre-filter → overall 직접 태깅 → 나머지 동적 샘플 → LLM
#    - trivial = 순수 감정 OR (그룹 지칭 + 감정)
#    - 동적 샘플링: <1K 전수, 1K~10K 10%, ≥10K 1000건
#    - 프롬프트에 topic description + representative_memos 포함
#    - checkpoint / resume
# ============================================================

# --------------------------------------------------
# 0. rule_map / topic_pool_group_map 로드
# --------------------------------------------------
rule_map = {
    (r["cate_1_depth"], r["cate_2_depth"], int(r["sc_measurement"])): r.asDict(recursive=True)
    for r in spark.table(RULE_PROFILE_TABLE).toLocalIterator()
}

topic_pool_group_map: Dict[tuple, List[Dict]] = {}
for row in spark.table(TOPIC_POOL_TABLE).toLocalIterator():
    key = (row["cate_1_depth"], row["cate_2_depth"], int(row["sc_measurement"]))
    reps = []
    try:
        reps = json.loads(row["representative_memos_json"]) if row["representative_memos_json"] else []
    except Exception:
        pass
    topic_pool_group_map.setdefault(key, []).append({
        "topic": row["topic"],
        "description": row["description"],
        "representative_memos": reps,
    })


# --------------------------------------------------
# 1. build_classify_messages
# --------------------------------------------------
def build_classify_messages(
    batch_rows: List[Dict[str, Any]],
    topic_pool_payload: List[Dict[str, Any]],
    rule_row: Dict[str, Any],
    cate_1_depth: str,
    cate_2_depth: str,
    sc_measurement: int,
) -> List[Dict[str, str]]:
    clue_keywords = json.loads(rule_row["clue_keywords_json"]) if rule_row["clue_keywords_json"] else []
    non_overall_examples = json.loads(rule_row["non_overall_examples_json"]) if rule_row["non_overall_examples_json"] else []
    category_patterns = get_feature_patterns(cate_1_depth, cate_2_depth)

    topic_info = []
    for t in topic_pool_payload:
        info = {"topic": t["topic"], "description": t["description"]}
        rep_memos = t.get("representative_memos", [])
        if rep_memos:
            info["examples"] = rep_memos[:3]
        topic_info.append(info)

    system = f"""
You are a VOC classifier for TV review topic classification.

Classify each memo into exactly one topic from the fixed topic list.
Every memo must belong to exactly one topic.

Rules:
- The task is to identify WHY the writer evaluated {cate_2_depth} as {sc_label(sc_measurement)}.
- Overall topic is '{clean_text(rule_row["overall_topic_label"])}'.
- {clean_text(rule_row["overall_usage_rule"])}
- {clean_text(rule_row["specific_reason_rule"])}
- clue keywords:
  {json.dumps(clue_keywords, ensure_ascii=False)}
- category fallback feature patterns:
  {json.dumps(category_patterns[:60], ensure_ascii=False)}
- non-overall examples:
  {json.dumps(non_overall_examples, ensure_ascii=False)}
- Do not invent any new topic.
- explanation must be a short Korean sentence.

Return only JSON:
{{
  "results": [
    {{
      "row_id": "",
      "topic": "",
      "explanation": ""
    }}
  ]
}}
"""
    user = (
        "Fixed topics (with description and example memos):\n"
        + json.dumps(topic_info, ensure_ascii=False)
        + "\n\nMemos:\n"
        + json.dumps(
            [{"row_id": str(r["_row_id"]), "memo": clean_text(r["memo"])} for r in batch_rows],
            ensure_ascii=False,
        )
    )
    return [
        {"role": "system", "content": clean_text(system)},
        {"role": "user", "content": clean_text(user)},
    ]


# --------------------------------------------------
# 2. build_reclassify_without_overall_message
# --------------------------------------------------
def build_reclassify_without_overall_message(
    row: Dict[str, Any],
    topic_pool_payload: List[Dict[str, Any]],
    rule_row: Dict[str, Any],
    cate_1_depth: str,
    cate_2_depth: str,
) -> List[Dict[str, str]]:
    overall_topic = clean_text(rule_row["overall_topic_label"])
    specific_topic_payload = [
        {"topic": t["topic"], "description": t["description"]}
        for t in topic_pool_payload if clean_text(t["topic"]) != overall_topic
    ]
    clue_keywords = json.loads(rule_row["clue_keywords_json"]) if rule_row["clue_keywords_json"] else []
    category_patterns = get_feature_patterns(cate_1_depth, cate_2_depth)

    system = f"""
You are a VOC classifier for TV review topic classification.

This memo was incorrectly over-generalized as '{overall_topic}'.
Reclassify it using only the non-general topics below.

Rules:
- Choose exactly one non-general topic.
- The memo contains a specific reason and must not remain as '{overall_topic}'.
- clue keywords:
  {json.dumps(clue_keywords, ensure_ascii=False)}
- category fallback feature patterns:
  {json.dumps(category_patterns[:60], ensure_ascii=False)}
- explanation must be a short Korean sentence.

Return only JSON:
{{
  "topic": "",
  "explanation": ""
}}
"""
    user = (
        "Allowed non-general topics:\n"
        + json.dumps(specific_topic_payload, ensure_ascii=False)
        + "\n\nMemo:\n"
        + json.dumps({"memo": clean_text(row["memo"])}, ensure_ascii=False)
    )
    return [
        {"role": "system", "content": clean_text(system)},
        {"role": "user", "content": clean_text(user)},
    ]


# --------------------------------------------------
# 3. 배치 LLM 분류 + overall 재분류 + 희소 주제 병합
# --------------------------------------------------
def classify_batch_and_merge(
    source_rows: List[Dict[str, Any]],
    topic_payload: List[Dict[str, Any]],
    rule_row: Dict[str, Any],
    c1: str, c2: str, sc: int,
    idx: int, total_groups: int,
) -> List[Dict[str, Any]]:
    clue_keywords = json.loads(rule_row["clue_keywords_json"]) if rule_row["clue_keywords_json"] else []
    generic_terms = json.loads(rule_row["generic_terms_json"]) if rule_row["generic_terms_json"] else []
    total_batches = (len(source_rows) + CLASSIFY_BATCH_SIZE - 1) // CLASSIFY_BATCH_SIZE

    classified_rows = []

    for batch_no, batch in enumerate(chunk_list(source_rows, CLASSIFY_BATCH_SIZE), start=1):
        batch_start_ts = time.time()
        print(f"  [BATCH] {batch_no}/{total_batches}, rows={len(batch)}")

        batch_result = call_llm(
            build_classify_messages(batch, topic_payload, rule_row, c1, c2, sc)
        )

        result_map = {
            str(item.get("row_id")): item
            for item in batch_result.get("results", [])
            if isinstance(item, dict)
        }

        for row in batch:
            mapped = result_map.get(str(row["_row_id"]), {})
            topic_raw = clean_text(mapped.get("topic"))
            explanation_raw = clean_text(mapped.get("explanation"))

            # overall 재분류
            if (
                topic_raw == clean_text(rule_row["overall_topic_label"])
                and has_specific_reason_for_category(row["memo"], clue_keywords, c1, c2)
                and not should_be_overall_for_category(row["memo"], clue_keywords, generic_terms, c1, c2)
            ):
                retry_result = call_llm(
                    build_reclassify_without_overall_message(row, topic_payload, rule_row, c1, c2)
                )
                retry_topic = clean_text(retry_result.get("topic"))
                retry_explanation = clean_text(retry_result.get("explanation"))
                if retry_topic:
                    topic_raw = retry_topic
                if retry_explanation:
                    explanation_raw = retry_explanation

            classified_rows.append({
                "_row_id": row["_row_id"],
                "cate_1_depth": c1,
                "cate_2_depth": c2,
                "sc_measurement": sc,
                "memo": row["memo"],
                "topic_raw": topic_raw,
                "explanation_raw": explanation_raw,
            })

        print(f"  [BATCH DONE] {batch_no}/{total_batches}, {round(time.time() - batch_start_ts, 2)}s")
        time.sleep(RATE_LIMIT_SECONDS)

    # 희소 주제 병합
    classified_group_df = spark.createDataFrame(pd.DataFrame(classified_rows))
    topic_stats_df = classified_group_df.groupBy("topic_raw").agg(F.count("*").alias("topic_cnt"))
    rare_topics = [r["topic_raw"] for r in topic_stats_df.where(F.col("topic_cnt") < F.lit(MIN_ROWS_PER_TOPIC)).collect()]
    rare_topics_sorted = sorted([t for t in rare_topics if clean_text(t)])
    rare_topic_label = f"기타({', '.join(rare_topics_sorted[:3])})" if rare_topics_sorted else "기타"

    topic_desc_map = {t["topic"]: t["description"] for t in topic_payload}

    final_rows = []
    for row in classified_group_df.toLocalIterator():
        rd = row.asDict(recursive=True)
        raw_topic = clean_text(rd["topic_raw"])
        raw_expl = clean_text(rd["explanation_raw"])

        if raw_topic in rare_topics_sorted:
            final_topic = rare_topic_label
            final_explanation = f"희소 주제 묶음: {raw_topic}" if raw_topic else "희소 주제 묶음"
        else:
            final_topic = raw_topic if raw_topic else "기타"
            final_explanation = raw_expl or topic_desc_map.get(final_topic, "")

        final_rows.append({
            "_row_id": rd["_row_id"],
            "cate_1_depth": rd["cate_1_depth"],
            "cate_2_depth": rd["cate_2_depth"],
            "sc_measurement": rd["sc_measurement"],
            "memo": rd["memo"],
            "topic": final_topic,
            "description": final_explanation,
        })

    return final_rows


# --------------------------------------------------
# 4. Main classify loop (통합 pre-filter)
# --------------------------------------------------
done_classify_groups = get_done_group_keys("classify_sample")
total_groups = len(target_groups)

print(f"[CLASSIFY] total_groups={total_groups}, done={len(done_classify_groups)}, pending={total_groups - len(done_classify_groups)}")

for idx, g in enumerate(target_groups, start=1):
    c1 = g["cate_1_depth"]
    c2 = g["cate_2_depth"]
    sc = int(g["sc_measurement"])
    key = (c1, c2, sc)

    if key in done_classify_groups:
        print(f"[SKIP] {idx}/{total_groups} already done - {key}")
        continue

    if key not in rule_map or key not in topic_pool_group_map:
        print(f"[SKIP] {idx}/{total_groups} missing stage1 data - {key}")
        continue

    print(f"\n[START] {idx}/{total_groups} - {key}")
    group_start_ts = time.time()

    try:
        rule_row = rule_map[key]
        topic_payload = topic_pool_group_map[key]
        clue_keywords = json.loads(rule_row["clue_keywords_json"]) if rule_row["clue_keywords_json"] else []

        # ==================================================
        # STEP 1. 그룹 지칭 단어 추출
        # ==================================================
        group_ref_words = get_group_reference_words(c1, c2, clue_keywords)
        print(f"  [REF WORDS] {sorted(list(group_ref_words))[:15]}...")

        # ==================================================
        # STEP 2. 전체 메모 로드 + rule pre-filter
        # ==================================================
        all_memos_df = load_group_all_memos_df(c1, c2, sc)
        total_count = all_memos_df.count()

        # closure 캐프처로 UDF 생성
        _ref_words = group_ref_words  # closure
        _trivial_udf = F.udf(
            lambda memo: is_trivial_memo(memo, _ref_words),
            T.BooleanType(),
        )
        tagged_df = all_memos_df.withColumn(
            "_is_trivial", _trivial_udf(F.col("memo"))
        )

        trivial_df = tagged_df.where(F.col("_is_trivial") == True).select("memo")
        non_trivial_df = tagged_df.where(F.col("_is_trivial") == False).select("memo")

        trivial_count = trivial_df.count()
        non_trivial_count = non_trivial_df.count()

        print(f"  [PRE-FILTER] total={total_count}, trivial={trivial_count}, non_trivial={non_trivial_count}")

        # ==================================================
        # STEP 3. Trivial 메모 → overall_topic_label 직접 태깅
        # ==================================================
        overall_topic_name = clean_text(rule_row["overall_topic_label"])
        trivial_rows = []
        for r in trivial_df.toLocalIterator():
            trivial_rows.append({
                "_row_id": -1,
                "cate_1_depth": c1,
                "cate_2_depth": c2,
                "sc_measurement": sc,
                "memo": r["memo"],
                "topic": overall_topic_name,
                "description": "순수 감정/지칭 표현 (rule 기반 자동 분류)",
            })

        # ==================================================
        # STEP 4. Non-trivial → 동적 샘플 → LLM 분류
        # ==================================================
        llm_final_rows = []
        if non_trivial_count > 0:
            sample_size = get_dynamic_sample_size(non_trivial_count)
            print(f"  [샘플] non_trivial {non_trivial_count}건 → {sample_size}건 샘플링")

            # deterministic sampling
            sampled_df = (
                non_trivial_df
                .withColumn(
                    "_rn",
                    F.row_number().over(
                        Window.orderBy(
                            F.md5(F.concat(
                                F.coalesce(F.col("memo"), F.lit("")),
                                F.lit(f"||{c1}||{c2}||{sc}||seed_20260420")
                            ))
                        )
                    )
                )
                .where(F.col("_rn") <= sample_size)
                .drop("_rn")
                .withColumn("_row_id", F.monotonically_increasing_id())
            ).cache()

            source_rows = [r.asDict(recursive=True) for r in sampled_df.toLocalIterator()]
            if source_rows:
                llm_final_rows = classify_batch_and_merge(
                    source_rows, topic_payload, rule_row, c1, c2, sc, idx, total_groups
                )
            sampled_df.unpersist()

        # ==================================================
        # STEP 5. 결합 저장 + checkpoint
        # ==================================================
        all_final_rows = trivial_rows + llm_final_rows

        if all_final_rows:
            delete_group_rows(DETAIL_TABLE, c1, c2, sc)
            append_table(spark.createDataFrame(pd.DataFrame(all_final_rows)), DETAIL_TABLE)

        log_progress(c1, c2, sc, "classify_sample", "done",
                     f"total={len(all_final_rows)}, trivial={len(trivial_rows)}, llm={len(llm_final_rows)}")

        elapsed_sec = round(time.time() - group_start_ts, 2)
        progress_pct = round(idx / total_groups * 100, 2)
        print(f"[DONE] {idx}/{total_groups} ({progress_pct}%) {len(all_final_rows)}rows (trivial={len(trivial_rows)}, llm={len(llm_final_rows)}), {elapsed_sec}s")

    except Exception as exc:
        log_failure(c1, c2, sc, "classify_sample", repr(exc))
        print(f"[FAILED] {idx}/{total_groups} error={repr(exc)}")

print("\n[CLASSIFY LOOP COMPLETE]")
display(spark.table(DETAIL_TABLE).limit(20))

In [0]:
# ============================================================
# 8) Rebuild Summary Table
# ============================================================

detail_df = spark.table(DETAIL_TABLE)

summary_df = (
    detail_df
    .groupBy("cate_1_depth", "cate_2_depth", "sc_measurement", "topic")
    .agg(F.count("*").alias("review_count"))
    .withColumn(
        "group_total_count",
        F.sum("review_count").over(
            Window.partitionBy("cate_1_depth", "cate_2_depth", "sc_measurement")
        ),
    )
    .withColumn("review_share", F.round(F.col("review_count") / F.col("group_total_count"), 4))
    .withColumn("review_share_pct", F.round(F.col("review_share") * 100, 2))
    .orderBy("cate_1_depth", "cate_2_depth", "sc_measurement", F.desc("review_count"))
)

summary_df.write.mode("overwrite").format("delta").saveAsTable(SUMMARY_TABLE)

print(f"[SUMMARY] saved to {SUMMARY_TABLE}")
display(summary_df)